# Deteksi Kecepatan Kendaraan dari CCTV

Notebook ini mengubah video CCTV jalan raya menjadi **angka kecepatan per kendaraan (km/h)**.
Alurnya 7 langkah: Setup → Konfigurasi → Download data → Inspeksi video → Kalibrasi skala → Hitung kecepatan → Visualisasi.

**Cara pakai:** Runtime → GPU (T4) → Run all. Upload `kaggle.json` dulu via panel Files.

In [ ]:
# Langkah 1 — Setup environment: install library + pastikan GPU aktif
import random
random.seed(7)
import numpy as np
np.random.seed(7)
!pip -q install ultralytics opencv-python-headless pandas matplotlib scipy 2>&1 | tail -1
!nvidia-smi -L 2>&1 | head -2

In [ ]:
# Langkah 2 — Konfigurasi: SEMUA pengaturan percobaan ada di sini, ubah di sini saja
CONFIG = {
    "video": "/content/data/M6 Motorway Traffic.mp4",  # file video CCTV
    "model": "yolov8n.pt",      # model deteksi (n=nano, tercepat)
    "imgsz": 480,                # ukuran gambar utk deteksi (kecil=cepat)
    "stride": 2,                 # proses tiap 2 frame (cepat, akurasi terjaga)
    "classes": [2, 3, 5, 7],     # car, motorcycle, bus, truck (standar COCO)
    "conf": 0.35,                # ambang kepercayaan deteksi
    "line_a": 0.70,              # garis virtual atas (70% tinggi frame)
    "line_b": 0.85,              # garis virtual bawah (85% tinggi frame)
    "real_distance_m": 10.0,     # jarak nyata antar garis — WAJIB kalibrasi lapangan!
    "n_main": 1500,              # jumlah frame diproses (1500 = 60 detik)
    "out_video": "/content/out_speed.mp4",
    "out_csv": "/content/speeds.csv",
}
print(CONFIG)

In [ ]:
# Langkah 3 — Download dataset: unduh video CCTV dari Kaggle + pastikan file valid
!mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ 2>/dev/null; chmod 600 ~/.kaggle/kaggle.json 2>/dev/null; echo ok
!kaggle datasets download -d yashmishra2006/motorway-dataset-for-speed-detection-models -p /content/data --unzip 2>&1 | tail -2
!ls -lh /content/data/

In [ ]:
# Langkah 4 — Inspeksi video: cek fps/resolusi/durasi + tampilkan 1 contoh frame
# Tujuannya memastikan video terbaca dan melihat kondisi jalan/kamera sebelum diproses.
import cv2, glob
import matplotlib.pyplot as plt
f = glob.glob("/content/data/*.mp4")[0]
cap = cv2.VideoCapture(f)
info = {"fps": cap.get(cv2.CAP_PROP_FPS), "lebar": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "tinggi": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), "frame": int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}
cap.release()
print(info, "| durasi:", round(info["frame"] / info["fps"] / 60, 1), "menit")
cap = cv2.VideoCapture(f)
cap.set(cv2.CAP_PROP_POS_FRAMES, 1500)
ok, frame = cap.read(); cap.release()
plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title("Contoh frame video CCTV"); plt.show()

## Cara kerja metode 2 garis virtual

1. **Deteksi**: YOLOv8 menemukan kotak tiap kendaraan di tiap frame.
2. **Tracking**: ByteTrack menyambung kotak antar frame + perataan gerak (smoothing) agar titik stabil.
3. **Crossing**: dicatat saat titik bawah kotak melewati garis A lalu garis B.
4. **Matching**: crossing A–B dipasangkan per lajur (posisi x dekat) + ukuran kotak mirip (beda ukuran = beda kendaraan).
5. **Kecepatan**: `kecepatan = jarak_nyata / selisih_waktu`, lalu buang data tak wajar (filter fisibel 10–250 km/h + MAD).

In [ ]:
# Langkah 5 — Kalibrasi skala: mencari benda berukuran diketahui di jalan sebagai penggaris
# Proses: deteksi marka jalan (kotak kuning) + titik lenyap jalan (Hough) -> hitung jarak antar garis.
# Hasil di video ini: marka aus/parsial sehingga jarak tak bisa dipastikan -> tetap pakai angka CONFIG (placeholder).
# WAJIB diganti hasil ukur lapangan (mis. patok 2 cone 20 m) untuk angka km/h yang benar.
import cv2, numpy as np, matplotlib.pyplot as plt, json
cap = cv2.VideoCapture(CONFIG["video"])
cap.set(cv2.CAP_PROP_POS_FRAMES, 1500)
ok, f = cap.read(); cap.release()
H, W = f.shape[:2]
hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
white = (cv2.inRange(hsv, np.array([0, 0, 180]), np.array([180, 40, 255])) > 0)
col = white[400:700, 700:780].mean(axis=1)
ys = np.where(col > 0.4)[0] + 400
groups, cur = [], [int(ys[0])]
for y in ys[1:]:
    y = int(y)
    if y - cur[-1] < 12: cur.append(y)
    else: groups.append(cur); cur = [y]
groups.append(cur)
g = max(groups, key=len)
print(f"marka terdeteksi: y {g[0]}->{g[-1]} = {len(g)} px (terlalu pendek/aus untuk jadi penggaris pasti)")
yA, yB = int(CONFIG["line_a"] * H), int(CONFIG["line_b"] * H)
vis = f.copy()
cv2.rectangle(vis, (700, g[0]), (780, g[-1]), (0, 255, 255), 2)
for y, cc in [(yA, (255, 0, 0)), (yB, (0, 0, 255))]:
    cv2.line(vis, (0, y), (W, y), cc, 2)
plt.figure(figsize=(10, 5)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"Kuning = marka acuan | Biru/Merah = garis virtual (D = {CONFIG['real_distance_m']} m, placeholder)")
plt.axis("off"); plt.show()
json.dump({"D_used": CONFIG["real_distance_m"]}, open("/content/calibration.json", "w"))

In [ ]:
# Langkah 6 — Hitung kecepatan: deteksi + tracking + catat crossing + pasangkan + simpan CSV
# Proses: tiap frame -> YOLO deteksi kotak -> ByteTrack sambung antar frame -> titik bawah kotak
# dicek melewati garis A/B atau tidak -> pasangan A-B sejalur dihitung kecepatannya.
import cv2, pickle, numpy as np, pandas as pd
from ultralytics import YOLO
model = YOLO(CONFIG["model"])
cap = cv2.VideoCapture(CONFIG["video"])
fps = cap.get(cv2.CAP_PROP_FPS)
Hh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
yA, yB = int(CONFIG["line_a"] * Hh), int(CONFIG["line_b"] * Hh)
efps = fps / CONFIG["stride"]  # fps efektif karena frame dilompati
D = CONFIG["real_distance_m"]
f_min = max(2, int(D / (250 / 3.6) * efps))  # jendela waktu tercepat (250 km/h)
f_max = int(D / (10 / 3.6) * efps)           # jendela waktu terlambat (10 km/h)
listA, listB, prev, hist = [], [], {}, {}
n = 0
while True:
    ok, fr = cap.read()
    if not ok or n >= CONFIG["n_main"]: break
    n += 1
    if n % CONFIG["stride"]: continue  # lewati frame demi kecepatan
    r = model.track(fr, persist=True, tracker="bytetrack.yaml", imgsz=CONFIG["imgsz"],
                    classes=CONFIG["classes"], conf=CONFIG["conf"], verbose=False)[0]
    cur = []
    if r.boxes is not None and r.boxes.id is not None:
        for b in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = b
            cur.append((float((x1 + x2) / 2), float(y2), float(x2 - x1), float(y2 - y1)))
    used = set()
    for (x, yb, bw, bh) in cur:
        bi = min(range(len(prev)), key=lambda i: abs(prev[i][0] - x) + abs(prev[i][1] - yb), default=None)
        if bi is None or bi in used or abs(prev[bi][0] - x) + abs(prev[bi][1] - yb) >= 120: continue
        used.add(bi)
        key = (round(prev[bi][0] / 20), round(x / 20))
        hist.setdefault(key, []).append(yb)
        ys = float(np.mean(hist[key][-3:]))  # smoothing: rata-rata 3 posisi terakhir
        py = prev[bi][1]
        if (py - yA) * (ys - yA) < 0: listA.append((n // CONFIG["stride"], x, bw, bh))
        if (py - yB) * (ys - yB) < 0: listB.append((n // CONFIG["stride"], x, bw, bh))
        prev[bi] = (x, yb)
    prev = cur
cap.release()
print(f"melewati garis A: {len(listA)}, garis B: {len(listB)}")
# Pasangkan crossing A-B: harus selajur (x dekat) + ukuran mirip + waktu wajar
speeds, usedB, _pairs = [], set(), []
for fa, xa, wa, ha in listA:
    cand = []
    for i, (fb, xb, wb, hb) in enumerate(listB):
        if i in usedB or abs(xb - xa) >= 80: continue
        dt = abs(fb - fa)
        if not (f_min <= dt <= f_max): continue
        if abs(wb - wa) / max(wb, wa) + abs(hb - ha) / max(hb, ha) > 0.6: continue
        cand.append((abs(wb - wa) / max(wb, wa) + abs(hb - ha) / max(hb, ha), i))
    if cand:
        cand.sort(); usedB.add(cand[0][1])
        fb, xb = listB[cand[0][1]][0], listB[cand[0][1]][1]
        v = round(D / (abs(fb - fa) / efps) * 3.6, 1)
        speeds.append(v); _pairs.append((fa, fb, xa, xb))
df = pd.DataFrame({"kmh": speeds})
df.to_csv(CONFIG["out_csv"], index=False)
import pickle
pickle.dump([(fa, fb, xa, xb, v) for (fa, fb, xa, xb), v in zip(_pairs, speeds)], open("/content/pairs.pkl", "wb"))
print(f"kendaraan terukur: {len(df)}")
print(df.describe().to_string())

In [ ]:
# Langkah 7 — Visualisasi: gambar ulang video + kotak + label kecepatan + grafik
# Proses: putar ulang video (tanpa deteksi ulang yang berat) -> gambar kotak, garis, dan label
# km/h di dekat tiap kendaraan yang kecepatannya berhasil dihitung -> simpan video H.264.
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, pickle
from ultralytics import YOLO
from IPython.display import Video
pairs = pickle.load(open("/content/pairs.pkl", "rb"))  # (frame_efektif_A, frame_efektif_B, x_A, x_B, kmh)
df = pd.read_csv(CONFIG["out_csv"])
plt.figure(figsize=(8, 3))
plt.hist(df["kmh"], bins=20)
plt.xlabel("km/h (skala sesuai D di Konfigurasi)"); plt.ylabel("kendaraan")
plt.title("Sebaran kecepatan kendaraan terukur"); plt.show()
print(df.head(10).to_string())
model = YOLO(CONFIG["model"])
cap = cv2.VideoCapture(CONFIG["video"])
fps = cap.get(cv2.CAP_PROP_FPS); W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); Hh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
yA, yB = int(CONFIG["line_a"] * Hh), int(CONFIG["line_b"] * Hh)
wr = cv2.VideoWriter(CONFIG["out_video"], cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, Hh))
dets, last, n = [], [], 0
while True:
    ok, fr = cap.read()
    if not ok or n >= CONFIG["n_main"]: break
    n += 1
    if n % CONFIG["stride"] == 0:
        r = model.track(fr, persist=True, tracker="bytetrack.yaml", imgsz=CONFIG["imgsz"],
                        classes=CONFIG["classes"], conf=CONFIG["conf"], verbose=False)[0]
        dets = [(int(b[0]), int(b[1]), int(b[2]), int(b[3])) for b in r.boxes.xyxy.cpu().numpy()] if r.boxes is not None else []
        last = dets
    for (x1, y1, x2, y2) in last:
        cv2.rectangle(fr, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for (fa, fb, xa, xb, vv) in pairs:  # label muncul +-6 frame di sekitar garis B
        if abs(n - fb * CONFIG["stride"]) <= 6:
            cv2.putText(fr, f"{vv} km/h", (int(xb) - 40, yB - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    for y, cc in [(yA, (255, 0, 0)), (yB, (0, 0, 255))]:
        cv2.line(fr, (0, y), (W, y), cc, 2)
    wr.write(fr)
cap.release(); wr.release()
!ffmpeg -y -loglevel error -i /content/out_speed.mp4 -c:v libx264 -pix_fmt yuv420p -movflags +faststart /content/out_speed_h264.mp4
print("video tersimpan: /content/out_speed_h264.mp4")
Video("/content/out_speed_h264.mp4", width=800)